# Four-in-a-row public data: what is there? (list only, nothing downloaded)

Lists the files in the public OSF projects and GitHub repositories of the Ma lab's four-in-a-row studies, with sizes:
- van Opheusden et al. 2023 (Nature), OSF `n2xjm`
- Ma, Phaneuf, van Opheusden, Ma & Hartley (developmental, ages 8–25), preprint `osf.io/d62rw` and any linked data project
- public GitHub repositories of the authors whose names suggest four-in-a-row or planning data

Run all, then copy the printed list into the chat. Downloading comes after we have seen what is there.

In [ ]:
import requests, json
S = requests.Session()
def get(u):
    r = S.get(u, timeout=60); r.raise_for_status(); return r.json()

def walk_osf(node, path=""):
    """All files under a node's OSF storage, with size in MB."""
    out, u = [], f"https://api.osf.io/v2/nodes/{node}/files/osfstorage/"
    stack = [(u, path)]
    while stack:
        url, p = stack.pop()
        while url:
            j = get(url)
            for it in j["data"]:
                a = it["attributes"]
                if a["kind"] == "folder":
                    stack.append((it["relationships"]["files"]["links"]["related"]["href"], p + a["name"] + "/"))
                else:
                    out.append((p + a["name"], (a.get("size") or 0) / 1e6, it["links"].get("download")))
            url = j["links"].get("next")
    return out

def show_node(node, depth=0):
    j = get(f"https://api.osf.io/v2/nodes/{node}/")
    print("  " * depth + f"== OSF {node}: {j['data']['attributes']['title']}")
    try:
        files = walk_osf(node)
        tot = sum(s for _, s, _ in files)
        print("  " * depth + f"   {len(files)} files, {tot:,.1f} MB total")
        for name, s, _ in sorted(files):
            print("  " * depth + f"   {s:10,.2f} MB  {name}")
    except Exception as e:
        print("  " * depth + "   (could not list files:", e, ")")
    try:
        for ch in get(f"https://api.osf.io/v2/nodes/{node}/children/")["data"]:
            show_node(ch["id"], depth + 1)
    except Exception as e:
        print("  " * depth + "   (no children listed:", e, ")")

show_node("n2xjm")

In [ ]:
# Developmental study preprint: linked data project(s)
try:
    p = get("https://api.osf.io/v2/preprints/d62rw/")
    a = p["data"]["attributes"]
    print("preprint:", a.get("title"))
    for k in ["data_links", "has_data_links", "why_no_data", "prereg_links"]:
        if k in a: print(f"  {k}: {a[k]}")
    rel = p["data"]["relationships"].get("node", {}).get("links", {}).get("related", {}).get("href")
    if rel:
        nid = rel.rstrip("/").split("/")[-1]
        print("  linked project:", nid); show_node(nid)
    else:
        print("  no linked OSF project in the preprint record")
except Exception as e:
    print("could not read preprint record:", e)

In [ ]:
# Public GitHub repositories of the authors / lab with planning or four-in-a-row in the name or description
for owner in ["WeiJiMaLab", "ili-ma", "basvanopheusden", "ikuperwajs", "cphaneuf", "daisylin", "xl1005"]:
    try:
        r = S.get(f"https://api.github.com/users/{owner}/repos?per_page=100", timeout=60)
        if r.status_code != 200:
            print(f"== {owner}: not found ({r.status_code})"); continue
        hits = [x for x in r.json() if any(k in (x["name"] + " " + (x["description"] or "")).lower()
                                            for k in ["row", "4iar", "fiar", "planning", "plan", "peak", "develop"])]
        print(f"== {owner}: {len(hits)} matching repos")
        for x in hits:
            print(f"   {x['size']/1e3:8.1f} MB  {x['html_url']}  {x['description'] or ''}")
    except Exception as e:
        print(owner, "error", e)